In [1]:
import pandas as pd
import numpy as np
import zipfile
import io
from google.colab import files

uploaded = files.upload()
zip_file = next(iter(uploaded))

with zipfile.ZipFile(io.BytesIO(uploaded[zip_file])) as z:
    csv_file = [f for f in z.namelist() if f.endswith(".csv")][0]
    df = pd.read_csv(z.open(csv_file))

print("Dataset Shape:", df.shape)
display(df.head())

print("\nMissing Values:")
print(df.isnull().sum())

Saving india_weather_data.csv.zip to india_weather_data.csv.zip
Dataset Shape: (131055, 18)


,date,latitude,longitude,wind_speed,cloud_cover,precipitation_probability,pressure_surface_level,dew_point,uv_index,heatwave,visibility,rainfall,solar_radiation,snowfall,max_temperature,min_temperature,max_humidity,min_humidity
0,2023-01-01 00:00:00,28.6139,77.2090,0.428064,56.787047,89.803147,1003.885666,29.282458,6.353398,0.0,0.827196,34.565602,NaN,0.0,104.379272,96.475944,25.221215,17.901661
1,2023-01-01 00:00:00,19.0760,72.8777,13.603933,36.321945,75.198203,500.000000,NaN,6.653509,0.0,8.040858,NaN,944.176384,0.0,38.198134,32.485543,86.025179,74.403003
2,2023-01-01 00:00:00,12.9716,77.5946,100.000000,49.585297,60.621663,1001.774831,31.567394,5.976996,0.0,6.776773,0.212155,451.964858,0.0,37.908737,NaN,65.569945,52.094434
3,2023-01-01 00:00:00,22.5726,88.3639,2.785259,21.462155,38.284893,993.455514,29.974869,7.757285,0.0,5.783869,0.389140,743.226528,0.0,33.755248,32.350385,69.260757,61.289844
4,2023-01-01 00:00:00,NaN,80.2707,9.399230,NaN,21.538080,1020.945313,25.815398,3.376130,0.0,8.133818,0.000000,493.243162,0.0,32.172688,27.963711,81.054374,75.400783



Missing Values:
date                         12853
latitude                     13145
longitude                    13287
wind_speed                   13169
cloud_cover                  12983
precipitation_probability    13234
pressure_surface_level       13249
dew_point                    12880
uv_index                     13318
heatwave                     13200
visibility                   13081
rainfall                     13049
solar_radiation              12996
snowfall                     13049
max_temperature              12875
min_temperature              13113
max_humidity                 12940
min_humidity                 13130
dtype: int64


In [2]:
data = df[["min_temperature", "max_temperature"]].dropna()

x = data["min_temperature"].values
y = data["max_temperature"].values

np.random.seed(42)
idx = np.random.permutation(len(data))
split = int(0.8 * len(data))

train, test = idx[:split], idx[split:]

x_train, y_train = x[train], y[train]
x_test, y_test = x[test], y[test]

slope = np.sum((x_train - x_train.mean()) * (y_train - y_train.mean())) / np.sum((x_train - x_train.mean()) ** 2)
intercept = y_train.mean() - slope * x_train.mean()

pred = intercept + slope * x_test

mae = np.mean(np.abs(y_test - pred))
rmse = np.sqrt(np.mean((y_test - pred) ** 2))
r2 = 1 - np.sum((y_test - pred) ** 2) / np.sum((y_test - y_test.mean()) ** 2)

print("Simple Linear Regression")
print("Slope:", slope)
print("Intercept:", intercept)
print("MAE:", mae)
print("RMSE:", rmse)
print("R2 Score:", r2)

missing = df["max_temperature"].isnull() & df["min_temperature"].notnull()
df["simple_prediction"] = df["max_temperature"]
df.loc[missing, "simple_prediction"] = (
    intercept + slope * df.loc[missing, "min_temperature"]
)

print("\nPredicted Missing Values:", missing.sum())
display(df.loc[missing, ["min_temperature", "simple_prediction"]].head())

Simple Linear Regression
Slope: 0.9724613182151397
Intercept: 5.76882737157154
MAE: 1.656713695453056
RMSE: 2.08261542579502
R2 Score: 0.945830293956329

Predicted Missing Values: 11581


,min_temperature,simple_prediction
12,18.897066,24.145493
13,27.331280,32.347440
39,7.921365,13.472049
48,33.691519,38.532527
62,15.808566,21.142047


In [5]:

features = ["min_temperature", "dew_point", "max_humidity", "min_humidity"]
target = "max_temperature"

data = df[features + [target]].dropna()

X = data[features].values
y = data[target].values

np.random.seed(42)
idx = np.random.permutation(len(data))
split = int(0.8 * len(data))

train, test = idx[:split], idx[split:]

X_train, y_train = X[train], y[train]
X_test, y_test = X[test], y[test]

X_train_b = np.column_stack((np.ones(len(X_train)), X_train))
X_test_b = np.column_stack((np.ones(len(X_test)), X_test))

b = np.linalg.pinv(X_train_b.T @ X_train_b) @ X_train_b.T @ y_train
pred = X_test_b @ b

mae = np.mean(np.abs(y_test - pred))
rmse = np.sqrt(np.mean((y_test - pred) ** 2))
r2 = 1 - np.sum((y_test - pred) ** 2) / np.sum((y_test - y_test.mean()) ** 2)

print("Multiple Linear Regression")
print("Coefficients:", b)
print("MAE:", mae)
print("RMSE:", rmse)
print("R2 Score:", r2)

full_data = df[features + [target]].dropna()
X_full = full_data[features].values
y_full = full_data[target].values
X_full_b = np.column_stack((np.ones(len(X_full)), X_full))

b_full = np.linalg.pinv(X_full_b.T @ X_full_b) @ X_full_b.T @ y_full

missing = df[target].isnull() & df[features].notnull().all(axis=1)
X_missing = df.loc[missing, features].values
X_missing_b = np.column_stack((np.ones(len(X_missing)), X_missing))

df["multiple_prediction"] = df[target]
df.loc[missing, "multiple_prediction"] = X_missing_b @ b_full

print("\nPredicted Missing Values:", missing.sum())
display(df.loc[missing, features + ["multiple_prediction"]].head())

Multiple Linear Regression
Coefficients: [5.37847062e+00 4.98931553e-01 4.86588466e-01 3.25601068e-04
 9.07768029e-05]
MAE: 1.4657890063913137
RMSE: 1.8020254723141254
R2 Score: 0.9600235544466317

Predicted Missing Values: 8462


,min_temperature,dew_point,max_humidity,min_humidity,multiple_prediction
12,18.897066,19.050157,50.608858,41.738684,24.093989
13,27.331280,24.158808,85.414253,75.335154,30.803706
39,7.921365,7.433578,34.690016,20.232238,12.941274
48,33.691519,35.395185,70.635071,63.372728,39.457034
62,15.808566,18.572081,42.228238,26.034356,22.314995


In [4]:
simple_mae = float(input("Simple Regression MAE: "))
multiple_mae = float(input("Multiple Regression MAE: "))

simple_rmse = float(input("Simple Regression RMSE: "))
multiple_rmse = float(input("Multiple Regression RMSE: "))

simple_r2 = float(input("Simple Regression R2: "))
multiple_r2 = float(input("Multiple Regression R2: "))

comparison = pd.DataFrame({
    "Metric": ["MAE", "RMSE", "R2 Score"],
    "Simple Linear Regression": [simple_mae, simple_rmse, simple_r2],
    "Multiple Linear Regression": [multiple_mae, multiple_rmse, multiple_r2]
})

display(comparison)

print("Better model based on MAE:",
      "Simple Linear Regression" if simple_mae < multiple_mae
      else "Multiple Linear Regression")

print("Better model based on R2:",
      "Simple Linear Regression" if simple_r2 > multiple_r2
      else "Multiple Linear Regression")

Simple Regression MAE: 1.656713695453056
Multiple Regression MAE: 1.4657890063913137
Simple Regression RMSE: 2.08261542579502
Multiple Regression RMSE: 1.8020254723141254
Simple Regression R2: 0.945830293956329
Multiple Regression R2: 0.9600235544466317


,Metric,Simple Linear Regression,Multiple Linear Regression
0,MAE,1.656714,1.465789
1,RMSE,2.082615,1.802025
2,R2 Score,0.945830,0.960024


Better model based on MAE: Multiple Linear Regression
Better model based on R2: Multiple Linear Regression
